# Session 1 – Do you really need a cluster?

**Goal:** Find out how far a single machine gets before we need distributed processing.
We run the *same* analysis on NYC Yellow Taxi data with three engines:

| Engine | Execution model |
|---|---|
| **pandas** | eager, (mostly) single-threaded, everything in RAM |
| **Polars** | lazy query plans, multi-threaded, columnar (Arrow) |
| **DuckDB** | SQL engine, multi-threaded, vectorized, can stream from disk |

Keep an eye on three things throughout: **runtime**, **memory**, and **how much data is actually read**.

In [ ]:
import os, time, glob, platform, psutil
import pandas as pd, polars as pl, duckdb, pyarrow as pa
import matplotlib.pyplot as plt

print("Python  ", platform.python_version())
print("CPU cores", os.cpu_count())
print(f"RAM      {psutil.virtual_memory().total/1e9:.1f} GB")
print("pandas", pd.__version__, "| polars", pl.__version__, "| duckdb", duckdb.__version__)

## 1. Get the data
Each month is ~50 MB of Parquet and ~3 million trips. Start with 6 months; we scale up later.
Run once in the terminal: `python get_data.py --months 6`

In [ ]:
FILES = sorted(glob.glob("../data/yellow_tripdata_*.parquet"))
print(len(FILES), "files,", f"{sum(os.path.getsize(f) for f in FILES)/1e6:.0f} MB on disk")

In [ ]:
# A tiny timing helper we reuse everywhere
def timed(label, fn):
    proc = psutil.Process()
    rss0 = proc.memory_info().rss
    t0 = time.perf_counter()
    result = fn()
    dt = time.perf_counter() - t0
    drss = (proc.memory_info().rss - rss0) / 1e6
    print(f"{label:<28} {dt:6.2f} s   Δ RSS {drss:7.0f} MB")
    return result, dt

## 2. The question
> **At which hour of the day do passengers tip the most (as % of the fare)?**
> Only credit-card payments (`payment_type == 1`), only fares > 0.

## 3. pandas – the baseline

In [ ]:
def run_pandas(files):
    df = pd.concat([pd.read_parquet(f) for f in files])          # loads ALL columns
    df = df[(df.payment_type == 1) & (df.fare_amount > 0)]
    df["hour"] = df.tpep_pickup_datetime.dt.hour
    df["tip_pct"] = df.tip_amount / df.fare_amount * 100
    return df.groupby("hour").agg(trips=("tip_pct", "size"), avg_tip_pct=("tip_pct", "mean")).reset_index()

res_pd, t_pd = timed("pandas", lambda: run_pandas(FILES))
res_pd.head()

**Your turn (5 min):** How many columns did pandas read, and how many did we actually need?
Try `pd.read_parquet(f, columns=[...])`. What changes?

## 4. Polars – lazy evaluation

In [ ]:
def polars_query(files):
    return (
        pl.scan_parquet(files)                                   # nothing is read yet!
          .filter((pl.col("payment_type") == 1) & (pl.col("fare_amount") > 0))
          .with_columns(hour=pl.col("tpep_pickup_datetime").dt.hour(),
                        tip_pct=pl.col("tip_amount") / pl.col("fare_amount") * 100)
          .group_by("hour")
          .agg(trips=pl.len(), avg_tip_pct=pl.col("tip_pct").mean())
          .sort("hour")
    )

print(polars_query(FILES).explain())                             # the optimized plan

Look at the plan: which columns end up in the scan (**projection pushdown**)? Where does the filter end up (**predicate pushdown**)?
Spark's optimizer (Session 2) does exactly the same thing.

In [ ]:
res_pl, t_pl = timed("polars (lazy)", lambda: polars_query(FILES).collect())
res_pl.head()

## 5. DuckDB – SQL straight on files

In [ ]:
con = duckdb.connect()
SQL = """
SELECT hour(tpep_pickup_datetime)            AS hour,
       count(*)                              AS trips,
       avg(tip_amount / fare_amount * 100)   AS avg_tip_pct
FROM read_parquet(?)
WHERE payment_type = 1 AND fare_amount > 0
GROUP BY hour ORDER BY hour
"""
res_dd, t_dd = timed("duckdb", lambda: con.execute(SQL, [FILES]).pl())
res_dd.head()

In [ ]:
# Same answer from all three engines?
import numpy as np
print(np.allclose(res_pd.sort_values("hour").avg_tip_pct.to_numpy(),
                  res_dd["avg_tip_pct"].to_numpy()))
print(f"speed-up vs pandas: polars {t_pd/t_pl:.1f}x | duckdb {t_pd/t_dd:.1f}x")

## 6. Why file formats matter: Parquet vs. CSV
Parquet is **columnar**, **compressed** and stores **min/max statistics** per row group.
CSV is none of that.

In [ ]:
one = FILES[0]
pl.read_parquet(one).write_csv("../data/one_month.csv")
print(f"Parquet {os.path.getsize(one)/1e6:6.0f} MB")
print(f"CSV     {os.path.getsize('../data/one_month.csv')/1e6:6.0f} MB")

q = "SELECT avg(tip_amount) FROM {src} WHERE payment_type = 1"
timed("duckdb on Parquet", lambda: con.sql(q.format(src=f"'{one}'")).fetchall())
timed("duckdb on CSV",     lambda: con.sql(q.format(src="'../data/one_month.csv'")).fetchall());

In [ ]:
# Peek inside the Parquet file: row groups and column statistics
meta = pa.parquet.ParquetFile(one).metadata
print(meta.num_row_groups, "row groups,", meta.num_columns, "columns")
col = meta.schema.names.index("fare_amount")
print(meta.row_group(0).column(col).statistics)   # min/max let engines skip whole row groups

## 7. Scaling experiment
Run the query on 1, 2, 4, … months. **Where does pandas break down on your machine?**
(In a 2-core / 8 GB Codespace, pandas usually struggles long before the others.)

In [ ]:
results = []
for n in sorted({n for n in [1, 2, 4, 8, 12] if n <= len(FILES)} | {len(FILES)}):
    subset = FILES[:n]
    print(f"--- {n} month(s)")
    for name, fn in [("pandas", lambda: run_pandas(subset)),
                     ("polars", lambda: polars_query(subset).collect()),
                     ("duckdb", lambda: con.execute(SQL, [subset]).pl())]:
        try:
            _, dt = timed(name, fn)
        except MemoryError:
            dt = float("nan"); print(name, "→ out of memory")
        results.append({"months": n, "engine": name, "seconds": dt})

res = pd.DataFrame(results).pivot(index="months", columns="engine", values="seconds")
res.plot(marker="o", title="Runtime vs. data volume", ylabel="seconds"); plt.show()
res

## 8. Discussion (group work, 15 min)
1. Why are Polars and DuckDB faster *even on the same single machine*? (Think: GIL, cores, columnar layout, lazy plans.)
2. Estimate: how many months of taxi data could this machine handle with DuckDB? What limits it – RAM, disk, or CPU?
3. Name **two situations** where a single machine is *not* enough, even with DuckDB. → That is where Spark (next session) comes in.
4. Big Data is not only *Volume*. Which of the other Vs did today's lab ignore completely?

> 💡 Questions like these – *explain why engine X beats engine Y* – are typical exam questions.